# 02 · 辅助数据准备

先运行01。默认使用缓存；仅首次下载时设置 RUN_NETWORK=True。

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if (ROOT / 'project' / 'pyproject.toml').exists(): ROOT = ROOT / 'project'
assert (ROOT / 'configs' / 'commodity_map.csv').exists(), 'Open the project folder first'
sys.path.insert(0, str(ROOT / 'src'))
from project.paths import ensure_directories
ensure_directories()
pd.set_option('display.max_columns', 14)


In [ ]:
from project.auxiliary import download_public, prepare_public, download_equities, prepare_equities
from project.acquisition import connect_wrds
RUN_NETWORK = False
if RUN_NETWORK:
    download_public(start_year=1986, end_year=2025)
    conn = connect_wrds()
    try:
        download_equities(conn)
    finally:
        conn.close()
prepared = prepare_public()
equity_monthly = prepare_equities()
display(pd.read_csv(ROOT / 'data/raw/auxiliary/download_status.csv'))
display(equity_monthly.tail())


## 全球股市波动率：分别保存两种定义

论文：先按日平均 G20 各国指数绝对收益，再按月取平均，最后一阶差分。这里选取 19 个主权国家的代表性本币指数，EU 不重复计入；这是可追溯的 **G20 proxy**，不是作者原指数集合。德国 DAX 为 performance index、部分国家历史更短、俄罗斯有停牌，必须查看覆盖。未前向填充停牌报价；超过七天的跨缺口收益留空。

Proposal：单一全球指数的月度实现波动率与论文不同。另存国家等权日收益的月度平方和开方作为 `vol_proposal_ew_proxy`，不称作真正全球指数。两者不可混用。VIX 保存月末 level 和实际观察日；z-score 的均值和标准差须仅在训练期估计。


In [ ]:
display(pd.read_csv(ROOT / 'data/interim/equity_coverage.csv'))
vix = pd.read_parquet(ROOT / 'data/processed/vix_monthly.parquet')
display(vix.tail())
assert (vix.observation_date <= vix.date).all()
assert (equity_monthly.countries_min <= 19).all()


## CFTC：测量日不等于公布日

Legacy futures-only 对应 non-commercial/commercial，不能与 disaggregated 管理基金口径拼接。原始数据保留所有市场，`configs/cftc_map.csv` 映射原论文品种。按报告日月末准备持仓水平、净投机头寸/OI、Proposal 非商业占比、hedging pressure、原文 Working's T。

`cftc_commodity_monthly_report_date.parquet` **只能用于报告日描述性研究**。Extension 1 预测必须提供官方 `report_date,release_date` 日历；不能机械用周二+3天（假日、停摆、补发会出错）。`align_releases` 使用实际公布日 as-of 对齐，并要求完整日历；未核对日历时不会产生可用于预测的持仓信号。

原论文使用 Working's T，Proposal 使用 non-commercial share，分别命名。AR(1) 创新应在选定样本/训练期估计，数据阶段保留水平，不提前用全样本回归生成预测解释变量。


In [ ]:
cot = pd.read_parquet(ROOT / 'data/processed/cftc_commodity_monthly_report_date.parquet')
display(cot.groupby('paper_code').agg(months=('report_date','size'),
                                    first=('report_date','min'),last=('report_date','max')))
display(cot[['paper_code','report_date','net_noncommercial_share','workings_t','availability_status']].tail())
release_calendar = ROOT / 'configs/cftc_release_dates.csv'
print('Official release calendar present:', release_calendar.exists())


## 宏观水平与状态

FRED 的 USREC、INDPRO、DTWEXBGS、TEDRATE、TB3MS、BAA、AAA 为本次快照的修订后数据。USREC 可用于事后经济周期描述；不能当实时可知状态。INDPRO 是美国工业产出，不能替代原论文 G7 工业产出。各序列原始频率不同，月度数据取月内最后非缺失观察，缺失不前向填充。


In [ ]:
macro = pd.read_parquet(ROOT / 'data/processed/macro_monthly_revised.parquet')
display(macro.tail())
display(macro.notna().sum().to_frame('nonmissing_months'))
